# Behavioral Metrics Analysis (Including Fiber Photometry Sessions)

This notebook calculates and analyzes behavioral metrics across all recording sessions, **including fiber photometry (FP) sessions**. The analysis is similar to `beh_metrics.ipynb` but includes additional FP session data from `hopkins_FP_session_assets.csv`.

## Import Libraries and Setup Paths

Import necessary libraries and set up paths. Note: this notebook uses the `cal_beh_metrics` function from `ani_session_processing.beh_mertics`.

In [1]:
import sys
import os
# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
_anchor = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.abspath(os.getcwd())
while _anchor != os.path.dirname(_anchor):
    _beh_ephys_root = os.path.join(_anchor, "code", "beh_ephys_analysis")
    if os.path.isdir(os.path.join(_beh_ephys_root, "utils")):
        if _beh_ephys_root in sys.path:
            sys.path.remove(_beh_ephys_root)
        sys.path.insert(0, _beh_ephys_root)
        break
    _anchor = os.path.dirname(_anchor)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
from pynwb import NWBFile, TimeSeries, NWBHDF5IO
import json
import seaborn as sns
from PyPDF2 import PdfMerger
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import re
from utils.beh_functions import session_dirs, parseSessionID, load_model_dv, makeSessionDF, get_session_tbl, get_unit_tbl, get_history_from_nwb, plot_session_glm
from utils.ephys_functions import*
from utils.ccf_utils import ccf_pts_convert_to_mm
import pickle
import scipy.stats as stats
import spikeinterface as si
import shutil
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import r2_score
from matplotlib import cm
import matplotlib.colors as mcolors
from joblib import Parallel, delayed
from utils.combine_tools import apply_qc
from scipy.stats import gaussian_kde
from ani_session_processing.beh_mertics import cal_beh_metrics
from utils.capsule_migration import CAPSULE_ROOT, capsule_directories

In [ ]:
%matplotlib inline

## Test Metric Calculation on a Single Session

Test the behavioral metrics calculation on one example session to verify it works.

In [ ]:
session = 'behavior_782394_2025-04-24_12-07-34'
model_name = 'stan_qLearning_5params'
cal_beh_metrics(session, model_name=model_name)

## Calculate Metrics for All Sessions (Including FP Sessions)

Load session IDs from three asset files (standard, Hopkins, and Hopkins FP), then compute behavioral metrics for all sessions in parallel.

In [ ]:
dfs = [pd.read_csv(CAPSULE_ROOT + '/code/data_management/session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_FP_session_assets.csv')]
df = pd.concat(dfs)
session_ids = df['session_id'].values
session_ids = [session_id for session_id in session_ids if isinstance(session_id, str)]  # filter only behavior sessions

def process_session(session):
    """
    """
    print(session)
    session_dir = session_dirs(session)
    if os.path.exists(os.path.join(session_dir['beh_fig_dir'], f'{session}.nwb')):
        try:
            cal_beh_metrics(session)
        except:
            print(f"Error processing session {session}")

# Use Parallel to process sessions in parallel
Parallel(n_jobs=12)(delayed(process_session)(session) for session in session_ids)
# process_session('behavior_716325_2024 -05-31_10-31-14')

## Merge All Session Metrics into Combined Dataframe

Load JSON files for each session and combine into a single DataFrame, including FP sessions.

In [ ]:
# Merge all JSON files into a dataframe
dfs = [pd.read_csv(CAPSULE_ROOT + '/code/data_management/session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_FP_session_assets.csv')]
df = pd.concat(dfs)
session_ids = df['session_id'].values
session_ids = [session_id for session_id in session_ids if isinstance(session_id, str)]  # filter only behavior sessions
combined_beh_sessions = []
for session in session_ids:
    session_dir = session_dirs(session)
    if os.path.exists(os.path.join(session_dir['beh_fig_dir'], f'{session}.nwb')):
        json_file_path = os.path.join(session_dir['beh_fig_dir'], f'{session}_beh_metrics.json')
        if os.path.exists(json_file_path):
            with open(json_file_path, 'r') as json_file:
                session_data = json.load(json_file)
            session_data['ani_id'] = session_dir['aniID']
            session_data['probe'] = df[df['session_id'] == session]['probe'].values[0] if 'probe' in df.columns else None
            session_data['session'] = session
            # if session_data['probe'] == 'tt':
            combined_beh_sessions.append(session_data)

## Calculate Derived Metrics and Save

Compute derived behavioral metrics (switch bias, lick bias, latency differences) and save to pickle file with FP sessions included.

In [ ]:
combined_beh_sessions['ani_id'].unique()

## Check Animal IDs Present in Dataset

Display unique animal IDs present in the combined dataset including FP sessions.

In [ ]:
combined_beh_file = os.path.join(str(capsule_directories()['derived_dir']) + '/combined/combined_session_tbl', 'combined_beh_sessions_with_FP.pkl')
with open(combined_beh_file, 'rb') as f:
    combined_beh_sessions = pickle.load(f)

## Load Combined Behavioral Metrics (with FP)

Load the saved combined behavioral metrics dataframe that includes fiber photometry sessions.

## Prepare Data for Scatter Plots

Clean the dataframe by removing non-numeric columns, keeping only metrics suitable for visualization.

In [ ]:
combined_beh_sessions_scatter = combined_beh_sessions.copy()
for col in combined_beh_sessions_scatter.columns:
    # Check if any value is a list, ndarray, or string
    if (combined_beh_sessions_scatter[col].apply(lambda x: isinstance(x, (list, np.ndarray, str))).any()) and col != 'ani_id' and col != 'probe' and col != 'session':
        combined_beh_sessions_scatter.drop(columns=[col], inplace=True)
# Save the cleaned dataframe
combined_beh_sessions_scatter.to_csv(str(capsule_directories()['derived_dir']) + '/combined/combined_session_tbl/combined_beh_sessions_scatter_FP.csv', index=False)

## Apply Quality Control Filters

Load QC criteria and filter sessions to identify those meeting quality standards.

In [ ]:
criteria_name = 'session_good'
# load constraints and data
with open(os.path.join(CAPSULE_ROOT + '/code/beh_ephys_analysis/session_combine/metrics', f'{criteria_name}.json'), 'r') as f:
    constraints = json.load(f)

# Apply constraints to the dataframe
combined_beh_sessions_scatter_filtered, combined_beh_sessions_scatter, fig, axes = apply_qc(combined_beh_sessions_scatter, constraints)

## Pairplot of All Behavioral Metrics

Create comprehensive pairwise scatter plot of all metrics, color-coded by animal ID (including FP sessions).

In [ ]:
from matplotlib import colormaps
from matplotlib.colors import to_hex
hue = 'ani_id'
cmap = colormaps['jet'] 

hue_values = combined_beh_sessions_scatter[hue].unique()
num_classes = len(hue_values)
palette = {
    val: to_hex(cmap(i / max(num_classes - 1, 1)))
    for i, val in enumerate(hue_values)
}
im = plt.scatter(combined_beh_sessions_scatter['lick_lat_diff'], combined_beh_sessions_scatter['var_lat_diff'], c = combined_beh_sessions_scatter['ani_id'].map(palette))
plt.colorbar(im)
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel('Lick Latency Difference')
plt.ylabel('Lick Variance Difference')

## Scatter Plot: Lick Latency vs Variance Difference

Visualize lick latency difference vs variance difference, color-coded by animal.

In [ ]:
g = sns.jointplot(
    data=combined_beh_sessions_scatter,
    x='lick_lat_diff_mode',
    y='var_lat_diff',
    hue='ani_id',
    palette=palette,
    kind='scatter'
)

# Remove existing legend
if g.ax_joint.legend_:
    g.ax_joint.legend_.remove()

# Create a smaller legend manually
g.ax_joint.legend(
    title='ani_id',
    fontsize=8,
    title_fontsize=8,
    markerscale=0.7,
    loc='upper right',
    frameon=False
)

g.ax_joint.set_xlim(-1, 1.5)   # example values
g.ax_joint.set_ylim(-4, 4)
g.ax_joint.axhline(0.75, color='gray', linestyle='--')
g.ax_joint.axhline(-0.75, color='gray', linestyle='--')
g.ax_joint.axvline(0.3, color='gray', linestyle='--')
g.ax_joint.axvline(-0.3, color='gray', linestyle='--')

In [ ]:
combined_beh_sessions_scatter[(combined_beh_sessions_scatter['lick_lat_diff_mode'].abs()>0.3) & (combined_beh_sessions_scatter['var_lat_diff'].abs()>0.75)]

## Compare Variance Metrics (Mean vs Mode-based)

Check consistency between variance metrics computed around mean vs mode.

## Identify Sessions with Strong Lick Bias

Display sessions exceeding bias thresholds (showing significant side bias in licking).

## Animal-Level Summary

Calculate and visualize average lick biases per animal across all their sessions (including FP sessions).

## Compare Mean vs Mode Lick Latency

Verify consistency between mean and mode lick latency difference metrics.

In [ ]:
g = sns.jointplot(
    data=combined_beh_sessions_scatter,
    x='var_lat_diff',
    y='var_lat_diff_mode',
    hue='ani_id',
    kind='scatter',
    palette=palette,
)
# Remove existing legend
if g.ax_joint.legend_:
    g.ax_joint.legend_.remove()

# Create a smaller legend manually
g.ax_joint.legend(
    title='ani_id',
    fontsize=8,
    title_fontsize=8,
    markerscale=0.7,
    loc='upper right',
    frameon=False
)

g.ax_joint.set_xlim(-4, 4)   # example values
g.ax_joint.set_ylim(-4, 4)
g.ax_joint.plot([-4, 4], [-4, 4], color='gray', linestyle='--')

## Overall Distribution

Joint plot showing overall distribution of lick biases across all sessions (including FP data).